In [1]:
import requests
from PIL import Image
import io
from IPython.display import display, Image as IPyImage

print("✅ کتابخانه‌ها با موفقیت بارگذاری شدند.")


✅ کتابخانه‌ها با موفقیت بارگذاری شدند.


In [2]:
# --- تنظیمات را اینجا وارد کنید ---
ESP32_IP = "10.64.102.138"  # آی‌پی برد ESP32 خود را اینجا بنویسید
IMAGE_PATH = "pic.jpg" # مسیر تصویر (مثلاً "C:/images/my_photo.jpg")

# ابعاد استاندارد برای مدل و LCD
TARGET_WIDTH = 320
TARGET_HEIGHT = 240
JPEG_QUALITY = 85

print(f"⚙️ تنظیمات اعمال شد:\n- IP: {ESP32_IP}\n- Image: {IMAGE_PATH}")

⚙️ تنظیمات اعمال شد:
- IP: 10.64.102.138
- Image: pic.jpg


In [3]:
try:
    # ۱. باز کردن تصویر و اطمینان از فرمت RGB
    img = Image.open(IMAGE_PATH).convert('RGB')
    
    # ۲. تغییر سایز با حفظ نسبت ابعاد
    img.thumbnail((TARGET_WIDTH, TARGET_HEIGHT), Image.Resampling.LANCZOS)
    
    # ۳. قرار دادن در مرکز بوم سیاه
    final_img = Image.new('RGB', (TARGET_WIDTH, TARGET_HEIGHT), (0, 0, 0))
    offset_x = (TARGET_WIDTH - img.width) // 2
    offset_y = (TARGET_HEIGHT - img.height) // 2
    final_img.paste(img, (offset_x, offset_y))
    
    # ۴. تبدیل به بایت‌های JPEG (با تضمین فرمت Baseline)
    buffer = io.BytesIO()
    # نکته کلیدی: progressive=False تضمین می‌کند که تصویر Baseline باشد
    final_img.save(buffer, format="JPEG", quality=85, progressive=False)
    image_data = buffer.getvalue()
    
    # ۵. بررسی هدر و نوع تصویر
    is_progressive = Image.open(IMAGE_PATH).info.get('progressive', 0)
    print(f"🔍 آیا تصویر اصلی Progressive بود؟ {bool(is_progressive)}")
    print(f"✅ تصویر پردازش و به Baseline تبدیل شد. حجم نهایی: {len(image_data)} بایت")
    print(f"🔍 ۴ بایت اول (Hex): {image_data[:4].hex()} (باید با ffd8 شروع شود)")
    
except Exception as e:
    print(f"❌ خطا در پردازش تصویر: {e}")

🔍 آیا تصویر اصلی Progressive بود؟ False
✅ تصویر پردازش و به Baseline تبدیل شد. حجم نهایی: 11808 بایت
🔍 ۴ بایت اول (Hex): ffd8ffe0 (باید با ffd8 شروع شود)


In [9]:
# انتخاب عملیات: "enroll" یا "recognize"
action = "recognize"  # <-- اینجا را تغییر دهید

url = f"http://{ESP32_IP}/api/face/{action}"
print(f"📡 در حال ارسال درخواست به: {url}")

try:
    headers = {"Content-Type": "image/jpeg"}
    # تایم‌اوت را روی ۱۵ ثانیه می‌گذاریم چون پردازش هوش مصنوعی کمی زمان می‌برد
    response = requests.post(url, data=image_data, headers=headers, timeout=15)
    
    print("-" * 40)
    print(f"🔹 کد وضعیت (Status Code): {response.status_code}")
    print(f"🔹 پاسخ سرور: {response.text}")
    print("-" * 40)
    
    if response.status_code == 200:
        print("✅ عملیات با موفقیت انجام شد!")
    else:
        print("⚠️ عملیات با خطا یا هشدار مواجه شد.")
        
except requests.exceptions.Timeout:
    print("❌ خطا: زمان درخواست به پایان رسید (Timeout). ممکن است برد مشغول پردازش باشد.")
except requests.exceptions.ConnectionError:
    print(f"❌ خطا: عدم توانایی در اتصال به {ESP32_IP}. لطفاً آی‌پی و اتصال وای‌فای را بررسی کنید.")
except Exception as e:
    print(f"❌ خطای غیرمنتظره: {e}")

📡 در حال ارسال درخواست به: http://10.64.102.138/api/face/recognize
----------------------------------------
🔹 کد وضعیت (Status Code): 200
🔹 پاسخ سرور: Access Granted! Welcome User ID: 2
----------------------------------------
✅ عملیات با موفقیت انجام شد!
